# Week 1 Day 5 — Final Model Validation, Deployment Readiness & Project Completion

## Adult Income Classification

**Primary metric:** F1-score  |  **Random state:** 42  |  **Split:** 60% train / 20% development / 20% untouched final test

## Task 1 — Final Model Validation

**Objective:** predict whether annual income is `>50K` (1) or `<=50K` (0).

In [ ]:
# Week 1 Day 5 — Final Model Validation, Deployment Readiness & Project Completion
## Adult Income Classification
**Primary metric:** F1-score | **Random state:** 42 | **Split:** 60% train / 20% development / 20% untouched final test
This notebook covers all six Day 5 tasks: final validation, error analysis, model interpretation, production inference, production inference. The final test set is not used for model selection or threshold selection.
## Task 1 — Final Model Validation
**Objective:** predict whether annual income is `>50K` (1) or `<=50K` (0).
The notebook first tries to load the Day 4 Joblib artifact. If it is unavailable, a reproducible fallback tuning workflow creates the artifact. In both cases, preprocessing is kept inside sklearn pipelines and the final test set remains locked until the final evaluation.
# WEEK 1 DAY 5 — COMPLETE EXECUTABLE NOTEBOOK
import os, sys, random, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)
from sklearn.calibration import calibration_curve
warnings.filterwarnings('ignore')
RANDOM_STATE=42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
RESULTS=Path('results')
FIGURES=Path('figures')
MODELS=Path('models')
for p in [RESULTS,FIGURES,MODELS]: p.mkdir(exist_ok=True)
print('Python:',sys.version.split()[0])
print('pandas:',pd.__version__,'numpy:',np.__version__,'scikit-learn:',sklearn.__version__,'joblib:',joblib.__version__)
# DATA LOADING
COLUMNS=['age','workclass','fnlwgt','education','education-num','marital-status','occupation', 'relationship','race','sex','capital-gain','capital-loss','hours-per-week','native-country','income']
def load_adult():
    candidates=['adult.data','adult.csv','adult_income.csv','adult_income_data.csv']
    for path in candidates:
    if not os.path.exists(path):
    continue
    if path.endswith('.data'):
    d=pd.read_csv(path,header=None,names=COLUMNS,skipinitialspace=True,na_values='?')
    else:
    raw=pd.read_csv(path,skipinitialspace=True,na_values='?')
    if set(COLUMNS).issubset(raw.columns): d=raw[COLUMNS].copy()
    elif raw.shape[1]==15: raw.columns=COLUMNS
    d=raw.copy()
    else: raise ValueError(f'Unexpected columns in {path}')
    source=path
    break
    else:
    from sklearn.datasets import fetch_openml
    d=fetch_openml('adult',version=2,as_frame=True).frame.copy()
    source='OpenML Adult v2'\n
    target='class' if 'class' in d.columns else 'income'
    d=d.rename(columns={target:'income','education_num':'education-num','marital_status':'marital-status',\n" 'capital_gain':'capital-gain','capital_loss':'capital-loss','hours_per_week':'hours-per-week',\n 'native_country':'native-country'})
    d=d[COLUMNS]
    for c in d.select_dtypes(include='object').columns:
    d[c]=d[c].astype(str).str.strip().replace({'?':np.nan,'nan':np.nan})
    d['income']=d['income'].astype(str).str.strip()
    d['target']=d['income'].map({'<=50K':0,'>50K':1,'<=50K.':0,'>50K.':1})
    dropped=int(d['target'].isna().sum())
    d=d.dropna(subset=['target']).copy()
    d['target']=d['target'].astype(int)
    d=d.drop(columns=['income'])
    return d,source,dropped

    df,data_source,dropped=load_adult()
    print('\nData source:',data_source,'| shape:',df.shape,'| dropped target rows:',dropped)
    numeric_features=['age','fnlwgt','education-num','capital-gain','capital-loss','hours-per-week']\ncategorical_features=['workclass','education','marital-status','occupation','relationship','race','sex','native-country']\n X=df.drop(columns='target')
    y=df['target'].astype(int)
    assert set(X.columns)==set(numeric_features+categorical_features)
    print('\nTarget distribution:')
    target_table=y.value_counts().sort_index().rename_axis('target').reset_index(name='count')\n"target_table['proportion']=(target_table['count']/len(y)).round(4)
    display(target_table)\n
    missing_table=pd.DataFrame({'Missing_Count':X.isna().sum(),'Missing_Percent':(X.isna().mean()*100).round(2)}).sort_values('Missing_Count',ascending=False)
    display(missing_table)
    target_table.to_csv(RESULTS/'target_distribution.csv',index=False)
    missing_table.to_csv(RESULTS/'missing_values.csv')
# 60/20/20 LEAKAGE-SAFE SPLIT
X_train_dev,X_test,y_train_dev,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=RANDOM_STATE)
X_train,X_dev,y_train,y_dev=train_test_split(X_train_dev,y_train_dev,test_size=.25,stratify=y_train_dev,random_state=RANDOM_STATE)
train_ids,dev_ids,test_ids=set(X_train.index),set(X_dev.index),set(X_test.index)
assert train_ids.isdisjoint(dev_ids) and train_ids.isdisjoint(test_ids) and dev_ids.isdisjoint(test_ids)
print('\nSplit sizes:',len(X_train),len(X_dev),len(X_test))
print('Leakage check: PASS — train/dev/test indices are disjoint.')
# PREPROCESSING — ALL INSIDE PIPELINE
try:
    enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False)
except TypeError: enc=OneHotEncoder(handle_unknown='ignore',sparse=False)\n
num_pipe=Pipeline([('imputer',SimpleImputer(strategy='median')),('scaler',StandardScaler())])
cat_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encoder',enc)])
preprocessor=ColumnTransformer([('numeric',num_pipe,numeric_features),('categorical',cat_pipe,categorical_features)])
# LOAD DAY 4 ARTIFACT — REQUIRED DAY 5 STEP
artifact_candidates=[Path('adult_income_final_pipeline.joblib'),MODELS/'adult_income_final_pipeline.joblib',Path('final_model.joblib'),MODELS/'final_model.joblib']\n
artifact_path=next((p for p in artifact_candidates if p.exists()),None)
"day4_loaded=False
artifact=None
saved_parameters={}\n if artifact_path:
artifact=joblib.load(artifact_path)
day4_loaded=True
if isinstance(artifact,dict):
" final_model=artifact.get('model')
selected_threshold=float(artifact.get('threshold',.50))
final_model_name=artifact.get('model_name','Saved Final Model')
saved_parameters=artifact.get('best_parameters',{})\n else:
final_model=artifact
selected_threshold=.50
final_model_name='Saved Pipeline' if final_model is None: raise ValueError('Day 4 artifact dictionary has no model key.')
print('\nDay 4 artifact loaded:',artifact_path)
print('Model:',final_model_name,'| threshold:',selected_threshold)
else:
print('\nDay 4 artifact not found — running reproducible fallback tuning.')
# FALLBACK TUNING: ONLY TRAINING DATA IS USED
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
models={
" 'Logistic Regression':Pipeline([('preprocessor',preprocessor),('model',LogisticRegression(max_iter=2000,random_state=RANDOM_STATE))]),\n" 'Random Forest':Pipeline([('preprocessor',preprocessor),('model',RandomForestClassifier(random_state=RANDOM_STATE,n_jobs=-1))]),\n" 'Gradient Boosting':Pipeline([('preprocessor',preprocessor),('model',GradientBoostingClassifier(random_state=RANDOM_STATE))])}\n params={
" 'Logistic Regression':{'model__C':np.logspace(-3,2,10),'model__class_weight':[None,'balanced'],'model__solver':['lbfgs']},\n" 'Random Forest':{'model__n_estimators':[100,200,300],'model__max_depth':[None,10,20,30],'model__min_samples_split':[2,5,10],'model__min_samples_leaf':[1,2,4],'model__max_features':['sqrt','log2',None]},\n" 'Gradient Boosting':{'model__n_estimators':[50,100,150,200],'model__learning_rate':[.01,.05,.1,.2],'model__max_depth':[2,3,4,5],'model__min_samples_split':[2,5,10],'model__min_samples_leaf':[1,2,4]}}\n searches={}
best_models={}
tuning_rows=[]
if not day4_loaded:
for name,est in models.items():
print('Tuning',name)
s=RandomizedSearchCV(est,params[name],n_iter=20,scoring='f1',cv=cv,random_state=RANDOM_STATE,n_jobs=-1,return_train_score=True)
s.fit(X_train,y_train)
searches[name]=s
best_models[name]=s.best_estimator_ tuning_rows.append({'Model':name,'Best_CV_F1':s.best_score_,'Best_Parameters':str(s.best_params_)})
print('Best CV F1:',round(s.best_score_,4),'| params:',s.best_params_)
tuning_summary=pd.DataFrame(tuning_rows).sort_values('Best_CV_F1',ascending=False)
tuning_summary.to_csv(RESULTS/'hyperparameter_search_summary.csv',index=False)
display(tuning_summary)
dev_rows=[]
for name,m in best_models.items():
p=m.predict_proba(X_dev)[:,1]
pred=(p>=.50).astype(int)
" dev_rows.append({'Model':name,'Threshold':.50,'Accuracy':accuracy_score(y_dev,pred),'Precision':precision_score(y_dev,pred,zero_division=0),'Recall':recall_score(y_dev,pred,zero_division=0),'F1':f1_score(y_dev,pred,zero_division=0),'ROC-AUC':roc_auc_score(y_dev,p),'PR-AUC':average_precision_score(y_dev,p),'Brier':brier_score_loss(y_dev,p)})\n development_comparison=pd.DataFrame(dev_rows).sort_values('F1',ascending=False)
development_comparison.to_csv(RESULTS/'development_model_comparison.csv',index=False)
display(development_comparison)
final_model_name=development_comparison.iloc[0]['Model']
final_model=best_models[final_model_name]
else:
# Preserve prior Day 4 comparison if present
otherwise create a transparent placeholder.
if (RESULTS/'development_model_comparison.csv').exists(): development_comparison=pd.read_csv(RESULTS/'development_model_comparison.csv')
else:
development_comparison=pd.DataFrame([{'Model':final_model_name,'Threshold':selected_threshold,'Accuracy':np.nan,'Precision':np.nan,'Recall':np.nan,'F1':np.nan,'ROC-AUC':np.nan,'PR-AUC':np.nan,'Brier':np.nan}])
development_comparison.to_csv(RESULTS/'development_model_comparison.csv',index=False)
# THRESHOLD: DEVELOPMENT ONLY
dev_probabilities=final_model.predict_proba(X_dev)[:,1]
if not day4_loaded:
rows=[]
for t in np.arange(.10,.91,.01):
pred=(dev_probabilities>=t).astype(int)
" rows.append({'Threshold':round(float(t),2),'Accuracy':accuracy_score(y_dev,pred),'Precision':precision_score(y_dev,pred,zero_division=0),'Recall':recall_score(y_dev,pred,zero_division=0),'F1':f1_score(y_dev,pred,zero_division=0)})\n" threshold_df=pd.DataFrame(rows)
best=threshold_df.sort_values(['F1','Precision','Recall','Threshold'],ascending=[False,False,False,True]).iloc[0]\n" selected_threshold=float(best['Threshold'])
threshold_df.to_csv(RESULTS/'threshold_analysis.csv',index=False)
display(threshold_df)\n" plt.figure(figsize=(8,5))
plt.plot(threshold_df.Threshold,threshold_df.Precision,label='Precision')
plt.plot(threshold_df.Threshold,threshold_df.Recall,label='Recall')
plt.plot(threshold_df.Threshold,threshold_df.F1,label='F1')
plt.axvline(selected_threshold,ls='--',label=f'Selected={selected_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Development Threshold Analysis')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURES/'threshold_analysis.png',dpi=150)
plt.show()\n else: print('Using stored Day 4 threshold
it is not retuned on test data.')
# PIPELINE VERIFICATION BEFORE TEST
probe=X_test.head(5)
probe_p=final_model.predict_proba(probe)[:,1]
probe_y=(probe_p>=selected_threshold).astype(int)
assert len(probe_p)==5 and np.all((probe_p>=0)&(probe_p<=1))
print('\nComplete pipeline verification: PASS')
# FINAL TEST — SINGLE LOCKED EVALUATION
test_probabilities=final_model.predict_proba(X_test)[:,1]
test_predictions=(test_probabilities>=selected_threshold).astype(int)
final_metrics={'Model':final_model_name,'Threshold':selected_threshold,'Accuracy':accuracy_score(y_test,test_predictions),'Precision':precision_score(y_test,test_predictions,zero_division=0),'Recall':recall_score(y_test,test_predictions,zero_division=0),'F1':f1_score(y_test,test_predictions,zero_division=0),'ROC-AUC':roc_auc_score(y_test,test_probabilities),'PR-AUC':average_precision_score(y_test,test_probabilities),'Brier':brier_score_loss(y_test,test_probabilities)}
final_test_df=pd.DataFrame([final_metrics])
final_test_df.to_csv(RESULTS/'final_test_metrics.csv',index=False)
print('\nFINAL TEST RESULTS')
display(final_test_df)
print(classification_report(y_test,test_predictions,target_names=['<=50K','>50K'],zero_division=0))
# CONFUSION MATRIX + FP/FN
cm=confusion_matrix(y_test,test_predictions)
tn,fp,fn,tp=cm.ravel()
pd.DataFrame([{'TN':tn,'FP':fp,'FN':fn,'TP':tp}]).to_csv(RESULTS/'confusion_matrix_summary.csv',index=False)
plt.figure(figsize=(6,5))
plt.imshow(cm)
plt.title('Confusion Matrix — Final Test Set')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks([0,1],['<=50K','>50K'])
plt.yticks([0,1],['<=50K','>50K'])
for i in range(2):
for j in range(2): plt.text(j,i,cm[i,j],ha='center',va='center')
plt.tight_layout()
plt.savefig(FIGURES/'confusion_matrix.png',dpi=150)
plt.show()
print('TN:',tn,'FP:',fp,'FN:',fn,'TP:',tp)
errors=X_test.copy()
errors['actual']=y_test.values
errors['predicted']=test_predictions
errors['probability']=test_probabilities
errors['error_type']=np.select([(errors.actual==0)&(errors.predicted==1),(errors.actual==1)&(errors.predicted==0)],['False Positive','False Negative'],default='Correct')
fp_df=errors[errors.error_type=='False Positive']
fn_df=errors[errors.error_type=='False Negative']
errors.to_csv(RESULTS/'test_error_analysis.csv',index=False)
fp_df.head(20).to_csv(RESULTS/'false_positive_samples.csv',index=False)
fn_df.head(20).to_csv(RESULTS/'false_negative_samples.csv',index=False)
print('False positives:',len(fp_df),'| False negatives:',len(fn_df))
display(fp_df.head(10))
display(fn_df.head(10))
# SUBGROUP + PATTERN ANALYSIS
def subgroup(Xg,yg,p,col,min_n=30):
z=Xg[[col]].copy()
z['actual']=np.asarray(yg)
z['pred']=(p>=selected_threshold).astype(int)
out=[]
for value,g in z.groupby(col,dropna=False):
if len(g)<min_n: continue out.append({'Feature':col,'Group':str(value),'N':len(g),'Accuracy':accuracy_score(g.actual,g.pred),'Precision':precision_score(g.actual,g.pred,zero_division=0),'Recall':recall_score(g.actual,g.pred,zero_division=0),'F1':f1_score(g.actual,g.pred,zero_division=0)})
return pd.DataFrame(out)
sg=pd.concat([subgroup(X_test,y_test,test_probabilities,c) for c in ['sex','race','marital-status','education','workclass']],ignore_index=True)
sg.to_csv(RESULTS/'subgroup_performance.csv',index=False)
display(sg.head(30))
pattern=[]
for c in ['sex','race','marital-status','education','workclass']:
z=errors.groupby(c,dropna=False).agg(N=('actual','size'),FP=('error_type',lambda s:(s=='False Positive').sum()),FN=('error_type',lambda s:(s=='False Negative').sum()),Errors=('error_type',lambda s:(s!='Correct').sum())).reset_index()
z['Error_Rate']=z.Errors/z.N
z['Feature']=c
pattern.append(z)
pattern_df=pd.concat(pattern,ignore_index=True)
pattern_df.to_csv(RESULTS/'error_pattern_summary.csv',index=False)
# MODEL INTERPRETATION
est=final_model.named_steps['model']
prep=final_model.named_steps['preprocessor']
names=prep.get_feature_names_out()
if hasattr(est,'coef_'):
coef=pd.DataFrame({'Feature':names,'Coefficient':est.coef_[0]})
coef['Absolute_Coefficient']=coef.Coefficient.abs()
coef=coef.sort_values('Absolute_Coefficient',ascending=False)
coef.to_csv(RESULTS/'logistic_regression_coefficients.csv',index=False)
display(coef.head(20))
top=pd.concat([coef.sort_values('Coefficient',ascending=False).head(10),coef.sort_values('Coefficient').head(10)]).drop_duplicates().sort_values('Coefficient')
plt.figure(figsize=(10,8))
plt.barh(top.Feature,top.Coefficient)
plt.axvline(0,ls='--')
plt.xlabel('Coefficient')
plt.ylabel('Transformed Feature')
plt.title('Top Logistic Regression Feature Effects')
plt.tight_layout()
plt.savefig(FIGURES/'feature_importance.png',dpi=150)
plt.show()
elif hasattr(est,'feature_importances_'):
imp=pd.DataFrame({'Feature':names,'Importance':est.feature_importances_}).sort_values('Importance',ascending=False)
imp.to_csv(RESULTS/'feature_importance_table.csv',index=False)
display(imp.head(20))
top=imp.head(20).iloc[::-1]
plt.figure(figsize=(10,8))
plt.barh(top.Feature,top.Importance)
plt.xlabel('Importance')
plt.title('Top Feature Importances')
plt.tight_layout()
plt.savefig(FIGURES/'feature_importance.png',dpi=150)
plt.show()
# LEARNING CURVE + CALIBRATION + ROC + PR
ts,train_scores,val_scores=learning_curve(final_model,X_train,y_train,cv=cv,scoring='f1',train_sizes=np.linspace(.10,1,5),n_jobs=-1)
trm,trs=train_scores.mean(1),train_scores.std(1)
vam,vas=val_scores.mean(1),val_scores.std(1)
plt.figure(figsize=(8,5))
plt.plot(ts,trm,marker='o',label='Training F1')
plt.plot(ts,vam,marker='o',label='Validation F1')
plt.xlabel('Training Examples')
plt.ylabel('F1')
plt.title('Learning Curve — Final Model')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURES/'learning_curve.png',dpi=150)
plt.show()
pd.DataFrame({'Training_Size':ts,'Training_F1':trm,'Validation_F1':vam}).to_csv(RESULTS/'learning_curve_summary.csv',index=False)
frac,meanp=calibration_curve(y_test,test_probabilities,n_bins=10,strategy='quantile')
plt.figure(figsize=(7,6))
plt.plot(meanp,frac,marker='o',label=f'Final model (Brier={final_metrics["Brier"]:.4f})')
plt.plot([0,1],[0,1],ls='--',label='Perfect Calibration')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Final Test Calibration Curve')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURES/'calibration_curve.png',dpi=150)
plt.show()
fpr,tpr,_=roc_curve(y_test,test_probabilities)
plt.figure(figsize=(7,6))
plt.plot(fpr,tpr,label=f'ROC-AUC={final_metrics["ROC-AUC"]:.3f}')
plt.plot([0,1],[0,1],ls='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Final Test ROC Curve')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURES/'final_test_roc.png',dpi=150)
plt.show()
pr,rc,_=precision_recall_curve(y_test,test_probabilities)
plt.figure(figsize=(7,6))
plt.plot(rc,pr,label=f'PR-AUC={final_metrics["PR-AUC"]:.3f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Final Test Precision-Recall Curve')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(FIGURES/'final_test_pr.png',dpi=150)
plt.show()
# FINAL METRICS TABLE
short=development_comparison.copy()
short['Evaluation_Set']='Development'
finalrow=final_test_df.copy()
finalrow['Evaluation_Set']='Untouched Test'
metrics_table=pd.concat([short,finalrow],ignore_index=True,sort=False)
metrics_table.to_csv(RESULTS/'final_metrics_table.csv',index=False)
display(metrics_table)
# PRODUCTION ARTIFACT + RELOAD VERIFICATION
artifact={'model':final_model,'threshold':float(selected_threshold),'model_name':final_model_name,'random_state':RANDOM_STATE,'numeric_features':numeric_features,'categorical_features':categorical_features,'best_parameters':saved_parameters if day4_loaded else (searches[final_model_name].best_params_ if final_model_name in searches else {}),'data_source':data_source,'python_version':sys.version.split()[0],'pandas_version':pd.__version__,'numpy_version':np.__version__,'scikit_learn_version':sklearn.__version__,'joblib_version':joblib.__version__}
joblib.dump(artifact,'adult_income_final_pipeline.joblib')
joblib.dump(artifact,MODELS/'adult_income_final_pipeline.joblib')
reloaded=joblib.load('adult_income_final_pipeline.joblib')
assert np.allclose(reloaded['model'].predict_proba(X_test.head(25))[:,1],final_model.predict_proba(X_test.head(25))[:,1])
print('Artifact reload verification: PASS')
def predict_income(new_data,artifact_path='adult_income_final_pipeline.joblib'):
a=joblib.load(artifact_path)
p=a['model'].predict_proba(new_data)[:,1]
pred=(p>=float(a['threshold'])).astype(int)
return pd.DataFrame({'probability_gt_50K':p,'prediction':pred,'predicted_income':np.where(pred==1,'>50K','<=50K')})
# 10 unseen/raw examples
examples=pd.DataFrame([ {'age':39,'workclass':'State-gov','fnlwgt':77516,'education':'Bachelors','education-num':13,'marital-status':'Never-married','occupation':'Adm-clerical','relationship':'Not-in-family','race':'White','sex':'Male','capital-gain':2174,'capital-loss':0,'hours-per-week':40,'native-country':'United-States'}, {'age':50,'workclass':'Private','fnlwgt':83311,'education':'Masters','education-num':14,'marital-status':'Married-civ-spouse','occupation':'Exec-managerial','relationship':'Husband','race':'White','sex':'Male','capital-gain':0,'capital-loss':0,'hours-per-week':50,'native-country':'United-States'}, {'age':28,'workclass':'Private','fnlwgt':123456,'education':'Bachelors','education-num':13,'marital-status':'Never-married','occupation':'Tech-support','relationship':'Not-in-family','race':'Asian-Pac-Islander','sex':'Female','capital-gain':0,'capital-loss':0,'hours-per-week':40,'native-country':'United-States'}, {'age':45,'workclass':'Self-emp-not-inc','fnlwgt':190000,'education':'HS-grad','education-num':9,'marital-status':'Married-civ-spouse','occupation':'Sales','relationship':'Husband','race':'White','sex':'Male','capital-gain':0,'capital-loss':0,'hours-per-week':60,'native-country':'United-States'}, {'age':23,'workclass':'Private','fnlwgt':150000,'education':'Some-college','education-num':10,'marital-status':'Never-married','occupation':'Sales','relationship':'Own-child','race':'White','sex':'Female','capital-gain':0,'capital-loss':0,'hours-per-week':35,'native-country':'United-States'}, {'age':60,'workclass':'Private','fnlwgt':210000,'education':'Masters','education-num':14,'marital-status':'Married-civ-spouse','occupation':'Prof-specialty','relationship':'Husband','race':'White','sex':'Male','capital-gain':0,'capital-loss':0,'hours-per-week':45,'native-country':'United-States'}, {'age':34,'workclass':'Private','fnlwgt':101010,'education':'Bachelors','education-num':13,'marital-status':'Divorced','occupation':'Prof-specialty','relationship':'Unmarried','race':'Black','sex':'Female','capital-gain':0,'capital-loss':0,'hours-per-week':40,'native-country':'United-States'}, {'age':52,'workclass':'Private','fnlwgt':180000,'education':'Doctorate','education-num':16,'marital-status':'Married-civ-spouse','occupation':'Prof-specialty','relationship':'Husband','race':'White','sex':'Male','capital-gain':99999,'capital-loss':0,'hours-per-week':50,'native-country':'United-States'}, {'age':31,'workclass':'Private','fnlwgt':140000,'education':'HS-grad','education-num':9,'marital-status':'Never-married','occupation':'Craft-repair','relationship':'Not-in-family','race':'White','sex':'Male','capital-gain':0,'capital-loss':0,'hours-per-week':40,'native-country':'United-States'}, {'age':42,'workclass':'Private','fnlwgt':160000,'education':'Bachelors','education-num':13,'marital-status':'Married-civ-spouse','occupation':'Exec-managerial','relationship':'Wife','race':'White','sex':'Female','capital-gain':0,'capital-loss':0,'hours-per-week':45,'native-country':'United-States'}])
inference_results=predict_income(examples)
inference_results.to_csv(RESULTS/'inference_examples.csv',index=False)
display(inference_results)
assert len(inference_results)==10
# REQUIREMENTS + INFERENCE SCRIPT
Path('inference.py').write_text('''import joblib, numpy as np, pandas as pd\n\ndef predict_income(new_data, artifact_path="adult_income_final_pipeline.joblib"):\n artifact=joblib.load(artifact_path)\n p=artifact["model"].predict_proba(new_data)[:,1]\n pred=(p>=float(artifact["threshold"])).astype(int)\n return pd.DataFrame({"probability_gt_50K":p,"prediction":pred,"predicted_income":np.where(pred==1,">50K","<=50K")})\n''',encoding='utf-8')
Path('requirements.txt').write_text(f'''numpy=={np.__version__}\npandas=={pd.__version__}\nscikit-learn=={sklearn.__version__}\njoblib=={joblib.__version__}\nmatplotlib=={__import__("matplotlib").__version__}\n''',encoding='utf-8')
print('\nDAY 5 COMPLETE — generated artifact, requirements, inference script, CSV results and PNG figures.')
## Final Submission Checklist
- [x] `adult_income_final_pipeline.joblib` - [x] Final Jupyter notebook - [x] Final metrics table - [x] Confusion matrix - [x] Learning curve - [x] Calibration curve - [x] ROC and Precision–Recall curves - [x] Feature interpretation visualization - [x] False-positive / false-negative samples - [x] Subgroup analysis - [x] Working inference function + 10 unseen examples - [x] `requirements.txt` - [x] Leakage verification - [x] Final test evaluation after model/threshold lock }],"metadata": {"kernelspec": { display_name": "Python 3 language": "python name": "python3

## Final Submission Checklist

- [x] `adult_income_final_pipeline.joblib`
- [x] Final Jupyter notebook
- [x] Final metrics table
- [x] Confusion matrix
- [x] Learning curve
- [x] Calibration curve
- [x] ROC and Precision–Recall curves
- [x] Feature interpretation visualization
- [x] False-positive / false-negative samples
- [x] Subgroup analysis
- [x] Working inference function + 10 unseen examples
- [x] `requirements.txt`
- [x] Leakage verification
- [x] Final test evaluation after model/threshold lock